In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path # manage paths for the project

In [ ]:
# define the path of the data
data_path = Path('..') / 'data' / 'raw' / 'educacionCol.csv'

In [ ]:
# Cargar dataset

df = pd.read_csv(data_path, low_memory=False)
df.head()

In [ ]:
df.tail()

In [ ]:
# exploracion inicial
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas \n")
df.info()

In [ ]:
# Analisis de valores nulos

nulos = df.isnull().sum()
nulos_porcentaje = (nulos / len(df)) * 100
df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_porcentaje})
print(df_nulos)

# Mostrar solo las columnas que tienen nulos
display(df_nulos[df_nulos['Nulos'] > 0].sort_values(by='Porcentaje (%)', ascending=False))

# No tenemos nulos

In [ ]:
total_duplicados = df.duplicated().sum()
print(f"Se encontraron {total_duplicados} filas exactamente duplicadas en el dataset.")

if total_duplicados > 0:
    # Mostrar una muestra de los duplicados
    display(df[df.duplicated(keep=False)].sort_values(by='Código SNIES delprograma').head(4))

In [ ]:
# Validacion de tipos de datos
display(df.dtypes)

# Total de matriculados esta como texto hay que corregirlo

In [ ]:
# 2. Forzar conversión a numérico, rellenar nulos con 0 y pasar a entero
df['Total Matriculados'] = pd.to_numeric(df['Total Matriculados'], errors='coerce').fillna(0).astype('int')

# --- Gráfico de Outliers ---
sns.boxplot(x=df['Total Matriculados'])

# --- Cuantificación de Outliers (Método IQR) ---
Q1 = df['Total Matriculados'].quantile(0.25)
Q3 = df['Total Matriculados'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
outliers = df[df['Total Matriculados'] > limite_superior]
print(f"Total de Outliers: {len(outliers)}")

In [ ]:
# Revisar si hay municipios escritos de diferentes formas (Mayúsculas vs Minúsculas o espacios al final)
municipios_unicos = df['Municipio de oferta del programa'].dropna().unique()
print(f"Total municipios únicos (sin limpiar): {len(municipios_unicos)}")

# Aplicando strip y upper para ver si el número se reduce (lo que indicaría inconsistencias de formato)
municipios_limpios = df['Municipio de oferta del programa'].dropna().astype(str).str.strip().str.upper().unique()
print(f"Total municipios únicos (limpiando espacios y mayúsculas): {len(municipios_limpios)}")

# Verificar longitudes raras en los años (deberían ser 4 caracteres)
años_inconsistentes = df[df['Año'].astype(str).str.len() != 4]['Año'].unique()
if len(años_inconsistentes) > 0:
    print(f"Años con formato inconsistente encontrados: {años_inconsistentes}")
else:
    print("La columna 'Año' tiene un formato consistente (4 dígitos).")

In [ ]:
# Calcular dimensiones
total_filas = len(df)
filas_unicas = len(df.drop_duplicates())
duplicados_exactos = total_filas - filas_unicas

# Preparar datos para el gráfico
datos_clones = pd.DataFrame({
    'Estado': ['1. Filas Originales (Crudo)', '2. Filas Únicas (Limpias)'],
    'Cantidad': [total_filas, filas_unicas]
})

# Configurar estilo
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")

# Crear gráfico de barras
ax = sns.barplot(x='Estado', y='Cantidad', data=datos_clones, palette=['#e74c3c', '#2ecc71'])

# Añadir etiquetas de datos encima de las barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.title('Impacto de Duplicados Exactos (Errores de Reporte del MEN)', fontsize=14, pad=15)
plt.ylabel('Volumen de Registros', fontsize=12)
plt.xlabel('')
plt.ylim(0, total_filas * 1.15) # Dar espacio arriba para el texto

# Añadir nota explicativa dentro del gráfico
plt.text(0.5, total_filas * 1.05, f'¡Se detectaron {duplicados_exactos:,} filas 100% clonadas!',
            ha='center', color='#c0392b', fontweight='bold', fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='#e74c3c', boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()

In [ ]:
cardinalidad = {
        'Programas Académicos (SNIES)': df['Código SNIES delprograma'].nunique(),
        'Municipios de Oferta': df['Código del Municipio(Programa)'].nunique(),
        'Instituciones (Universidades)': df['Institución de Educación Superior (IES)'].nunique(),
        'Departamentos': df['Departamento de oferta del programa'].nunique(),
        'Niveles de Formación': df['Id_Nivel_Formacion'].nunique(),
        'Metodologías': df['Id_Metodologia'].nunique(),
        'Géneros': df['Id Género'].nunique()
}

# Convertir a DataFrame y ordenar de mayor a menor
df_cardinalidad = pd.DataFrame(list(cardinalidad.items()), columns=['Entidad / Dimensión', 'Valores Únicos'])
df_cardinalidad = df_cardinalidad.sort_values('Valores Únicos', ascending=True)

# Configurar gráfico
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# Gráfico de barras horizontales (ideal para etiquetas largas)
ax = sns.barplot(x='Valores Únicos', y='Entidad / Dimensión', data=df_cardinalidad, palette='viridis')

# Añadir etiquetas de datos a la derecha de las barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0),
                textcoords='offset points',
                fontsize=11, fontweight='bold')

# Escala logarítmica en el eje X para que los valores pequeños (Género=2) se vean junto a los gigantes (Programas=10,000)
plt.xscale('log')

plt.title('Cardinalidad de Entidades (Justificación del Modelo Dimensional)', fontsize=14, pad=15)
plt.xlabel('Cantidad de Valores Únicos (Escala Logarítmica)', fontsize=12)
plt.ylabel('Dimensiones de Negocio', fontsize=12)

# Añadir nota explicativa
nota = ("La alta varianza entre Programas (>8,000) y\n"
        "Géneros (2) justifica la separación en múltiples\n"
        "Tablas de Dimensiones para evitar redundancia.")
plt.text(10, 5, nota, fontsize=10, style='italic',
        bbox=dict(facecolor='#f1f2f6', alpha=0.9, edgecolor='gray', boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()

In [ ]:
# tranformacion temporal
df.columns = df.columns.str.lower()

In [ ]:
text_colummns = df.select_dtypes(include=['object', 
                                        'string']).columns

for column in text_colummns:
    df[column] = df[column].str.lower()
    
df.head(100)

In [ ]:
df["departamento de domicilio de la ies"].unique()

In [ ]:
df["departamento de oferta del programa"].unique()

In [ ]:
df["id género"].unique() # por si depronto hay valores que no sean 1 y 2